# 00c: Prompt Engineering for Hydrology Programming

**📖 Self-Study Notebook** — Work through this material during breaks or after hours at your own pace.

This notebook teaches you how to write effective prompts for AI coding assistants when working on scientific programming tasks in hydrology. Better prompts lead to better code on the first try, which means less time debugging and more time doing science.

**Learning objectives:**
- Master five prompt patterns that improve AI output for hydrology code
- Compare vague vs. structured prompts and see the difference in output quality
- Practice iterative prompt refinement on a realistic hydrology task
- Understand how adding physical constraints to prompts improves correctness

> **Note:** All AI-generated code examples in this notebook were captured ahead of time. No live AI API calls are made. This notebook works without any AI tool installed.

---
## 1. Five Prompt Patterns for Hydrology Programming

When asking an AI assistant to write scientific code, the quality of your prompt directly determines the quality of the output. Below are five reusable patterns, each illustrated with a hydrology example.

### Pattern A: Specifying Data Types and Shapes

Tell the AI exactly what goes in and what comes out — data types, array shapes, and index structures.

**Why it matters:** AI assistants frequently guess wrong about whether your data is a list, a NumPy array, or a Pandas DataFrame. Being explicit avoids an entire class of errors.

#### Hydrology example

**Weak prompt:**
> "Write a function to calculate daily mean streamflow from 15-minute data."

**Strong prompt using Pattern A:**
> "Write a Python function that takes a pandas DataFrame with a DatetimeIndex at 15-minute intervals and a column named 'discharge_cms' (float64, units: cubic meters per second). Return a new DataFrame with a daily DatetimeIndex and a column 'mean_daily_discharge_cms' containing the daily mean of the input discharge values."

#### AI output from the strong prompt:

In [ ]:
# AI-generated code from Pattern A prompt
import pandas as pd
import numpy as np


def daily_mean_streamflow(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate daily mean streamflow from 15-minute interval data.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with a DatetimeIndex at 15-minute intervals and a column
        'discharge_cms' (float64) in cubic meters per second.

    Returns
    -------
    pd.DataFrame
        DataFrame with a daily DatetimeIndex and column
        'mean_daily_discharge_cms'.
    """
    daily = df["discharge_cms"].resample("D").mean()
    return daily.to_frame(name="mean_daily_discharge_cms")

In [ ]:
# Quick verification
dates = pd.date_range("2023-06-01", periods=96, freq="15min")  # 1 day of 15-min data
test_df = pd.DataFrame({"discharge_cms": np.full(96, 5.0)}, index=dates)
result = daily_mean_streamflow(test_df)
print(result)
assert result["mean_daily_discharge_cms"].iloc[0] == 5.0, "Daily mean should be 5.0"
print("✅ Pattern A output verified.")

**Takeaway:** By specifying the DataFrame structure, column name, data type, and units, the AI produced a clean function with the correct resample logic on the first try.

### Pattern B: Including Units and Physical Constraints

Always state the units for every quantity and any physical constraints the code must respect.

**Why it matters:** AI models have no understanding of physics. Without explicit constraints, they may produce code that accepts negative hydraulic conductivity or returns heads above the land surface.

#### Hydrology example

**Weak prompt:**
> "Write a function to compute Darcy velocity."

**Strong prompt using Pattern B:**
> "Write a Python function that computes Darcy velocity (specific discharge) using Darcy's Law: q = -K * dh/dl. Parameters: K (hydraulic conductivity in m/day, must be positive), dh (head difference in meters), dl (flow path length in meters, must be positive). Return q in m/day. Raise a ValueError if K or dl is not positive."

#### AI output from the strong prompt:

In [ ]:
# AI-generated code from Pattern B prompt
def darcy_velocity(K: float, dh: float, dl: float) -> float:
    """Compute Darcy velocity (specific discharge).

    Parameters
    ----------
    K : float
        Hydraulic conductivity in m/day. Must be positive.
    dh : float
        Head difference in meters (h_upstream - h_downstream).
    dl : float
        Flow path length in meters. Must be positive.

    Returns
    -------
    float
        Darcy velocity (specific discharge) in m/day.

    Raises
    ------
    ValueError
        If K or dl is not positive.
    """
    if K <= 0:
        raise ValueError(f"Hydraulic conductivity must be positive, got {K}")
    if dl <= 0:
        raise ValueError(f"Flow path length must be positive, got {dl}")
    return -K * (dh / dl)

In [ ]:
# Verification
# K=10 m/day, head drops 5 m over 100 m => q = -10 * (5/100) = -0.5 m/day
# (negative because flow is in the direction of decreasing head)
q = darcy_velocity(K=10.0, dh=5.0, dl=100.0)
print(f"Darcy velocity: {q:.2f} m/day")
assert abs(q - (-0.5)) < 1e-10, f"Expected -0.5, got {q}"

# Verify constraint enforcement
try:
    darcy_velocity(K=-1.0, dh=5.0, dl=100.0)
    print("❌ Should have raised ValueError for negative K")
except ValueError as e:
    print(f"✅ Correctly rejected negative K: {e}")

**Takeaway:** Including units and constraints in the prompt caused the AI to add input validation that would otherwise be missing. Without the constraint language, AI-generated Darcy's Law implementations almost never check for negative K.

### Pattern C: Requesting Specific Libraries

Tell the AI which libraries to use (and which to avoid).

**Why it matters:** AI assistants often default to basic Python or pick whichever library appeared most in their training data. For scientific work, you usually want specific libraries (for example, NumPy for arrays, Pandas for time series, xarray for gridded data).

#### Hydrology example

**Weak prompt:**
> "Read a CSV of streamflow data and compute monthly statistics."

**Strong prompt using Pattern C:**
> "Using pandas (not raw Python file I/O), read a CSV file with columns 'date' (YYYY-MM-DD) and 'streamflow_cms' (float). Parse 'date' as a DatetimeIndex. Compute monthly mean, min, and max streamflow. Return a DataFrame with columns 'mean_cms', 'min_cms', 'max_cms' and a monthly PeriodIndex."

#### AI output from the strong prompt:

In [ ]:
# AI-generated code from Pattern C prompt
def monthly_streamflow_stats(csv_path: str) -> pd.DataFrame:
    """Read streamflow CSV and compute monthly statistics.

    Parameters
    ----------
    csv_path : str
        Path to CSV with columns 'date' (YYYY-MM-DD) and
        'streamflow_cms' (float).

    Returns
    -------
    pd.DataFrame
        Monthly statistics with PeriodIndex and columns
        'mean_cms', 'min_cms', 'max_cms'.
    """
    df = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")
    monthly = df["streamflow_cms"].resample("ME").agg(
        mean_cms="mean",
        min_cms="min",
        max_cms="max",
    )
    monthly.index = monthly.index.to_period("M")
    return monthly

In [ ]:
# Verification with synthetic data (no file needed)
import io

csv_data = """date,streamflow_cms
2023-01-01,10.0
2023-01-15,20.0
2023-01-31,15.0
2023-02-01,8.0
2023-02-14,12.0
2023-02-28,10.0
"""

# Write to a temporary approach — use StringIO to avoid file I/O
import tempfile, os

with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False) as f:
    f.write(csv_data)
    tmp_path = f.name

result = monthly_streamflow_stats(tmp_path)
os.unlink(tmp_path)

print(result)
assert abs(result.loc["2023-01", "mean_cms"] - 15.0) < 1e-10
assert result.loc["2023-01", "min_cms"] == 10.0
assert result.loc["2023-01", "max_cms"] == 20.0
print("\n✅ Pattern C output verified.")

**Takeaway:** Specifying "pandas, not raw Python" ensured the AI used `read_csv` with `parse_dates` and `resample` instead of writing manual date-parsing loops.

### Pattern D: Providing Example Input-Output Pairs

Give the AI a concrete example of what goes in and what should come out.

**Why it matters:** Examples make your intent much clearer than prose descriptions. They also give you a built-in test case.

#### Hydrology example

**Weak prompt:**
> "Write a function to classify streamflow into low, medium, and high categories."

**Strong prompt using Pattern D:**
> "Write a Python function that classifies daily streamflow values into categories based on percentile thresholds. Input: a 1D numpy array of daily streamflow in cms. Output: a 1D numpy array of strings with the same length. Classification rules: values below the 25th percentile → 'low', values between 25th and 75th percentile → 'medium', values above 75th percentile → 'high'. Example: given input `[1.5, 2.3, 0.8, 5.1, 3.0, 0.5, 4.2, 2.8]`, the 25th percentile is ~1.325 and 75th percentile is ~3.6, so expected output is `['medium', 'medium', 'low', 'high', 'medium', 'low', 'high', 'medium']`."

#### AI output from the strong prompt:

In [ ]:
# AI-generated code from Pattern D prompt
def classify_streamflow(discharge: np.ndarray) -> np.ndarray:
    """Classify daily streamflow into low, medium, and high categories.

    Parameters
    ----------
    discharge : np.ndarray
        1D array of daily streamflow values in cms.

    Returns
    -------
    np.ndarray
        1D array of strings: 'low', 'medium', or 'high'.
    """
    p25 = np.percentile(discharge, 25)
    p75 = np.percentile(discharge, 75)

    categories = np.where(
        discharge < p25,
        "low",
        np.where(discharge > p75, "high", "medium"),
    )
    return categories

In [ ]:
# Verification using the example from the prompt
test_input = np.array([1.5, 2.3, 0.8, 5.1, 3.0, 0.5, 4.2, 2.8])
expected = np.array(["medium", "medium", "low", "high", "medium", "low", "high", "medium"])

result = classify_streamflow(test_input)
print(f"Input:    {test_input}")
print(f"Output:   {result}")
print(f"Expected: {expected}")
assert np.array_equal(result, expected), f"Mismatch: {result} != {expected}"
print("\n✅ Pattern D output verified.")

**Takeaway:** The example I/O pair removed all ambiguity about boundary handling (strict `<` vs. `<=`) and gave us an instant test case to verify the output.

### Pattern E: Requesting Documentation

Ask the AI to include docstrings, type hints, and comments.

**Why it matters:** Documentation forces the AI to "think through" the function's contract. Functions with good docstrings tend to have fewer bugs because the AI has to reconcile the description with the implementation.

#### Hydrology example

**Weak prompt:**
> "Write a function to estimate evapotranspiration."

**Strong prompt using Pattern E:**
> "Write a Python function that estimates reference evapotranspiration (ET₀) using the Hargreaves method: ET0 = 0.0023 * (T_mean + 17.8) * (T_max - T_min)^0.5 * Ra. Parameters: T_mean, T_max, T_min (all in °C), Ra (extraterrestrial radiation in mm/day). Return ET0 in mm/day. Include a NumPy-style docstring with Parameters and Returns sections. Add type hints. Add inline comments explaining each step of the calculation."

#### AI output from the strong prompt:

In [ ]:
# AI-generated code from Pattern E prompt
def hargreaves_et0(
    T_mean: float, T_max: float, T_min: float, Ra: float
) -> float:
    """Estimate reference evapotranspiration using the Hargreaves method.

    Implements the Hargreaves (1985) equation:
        ET0 = 0.0023 * (T_mean + 17.8) * (T_max - T_min)^0.5 * Ra

    Parameters
    ----------
    T_mean : float
        Mean daily air temperature in degrees Celsius.
    T_max : float
        Maximum daily air temperature in degrees Celsius.
    T_min : float
        Minimum daily air temperature in degrees Celsius.
    Ra : float
        Extraterrestrial radiation in mm/day.

    Returns
    -------
    float
        Reference evapotranspiration (ET0) in mm/day.
    """
    # Temperature range (diurnal temperature difference)
    T_range = T_max - T_min

    # Hargreaves empirical coefficient
    coeff = 0.0023

    # Temperature adjustment factor
    T_adj = T_mean + 17.8

    # Compute ET0 using the Hargreaves equation
    et0 = coeff * T_adj * (T_range ** 0.5) * Ra

    return et0

In [ ]:
# Verification with a hand-calculated example
# T_mean=20°C, T_max=30°C, T_min=10°C, Ra=15 mm/day
# ET0 = 0.0023 * (20 + 17.8) * (30 - 10)^0.5 * 15
#     = 0.0023 * 37.8 * 4.4721 * 15
#     = 0.0023 * 37.8 * 67.082
#     = 0.0023 * 2535.7
#     ≈ 5.832
et0 = hargreaves_et0(T_mean=20.0, T_max=30.0, T_min=10.0, Ra=15.0)
expected_et0 = 0.0023 * (20.0 + 17.8) * (20.0 ** 0.5) * 15.0
print(f"Computed ET0: {et0:.4f} mm/day")
print(f"Expected ET0: {expected_et0:.4f} mm/day")
assert abs(et0 - expected_et0) < 1e-10, f"Mismatch: {et0} != {expected_et0}"
print("\n✅ Pattern E output verified.")

**Takeaway:** Requesting documentation and type hints produced a well-structured function with clear inline comments. The docstring makes it easy for a reviewer to verify the implementation matches the equation.

---
## 2. Comparative Exercise: Vague vs. Structured Prompts

The best way to appreciate prompt quality is to see the same task solved with a vague prompt and a structured prompt side by side. Below we ask an AI assistant to **read and plot USGS streamflow data** using two very different prompts.

### Vague Prompt

> "Plot some USGS streamflow data."

#### AI output from the vague prompt:

In [ ]:
# AI-generated code from the VAGUE prompt
# "Plot some USGS streamflow data."
import matplotlib.pyplot as plt

# The AI invented a URL that may not work and used no date parsing
data = [120, 115, 130, 125, 140, 135, 128, 122, 118, 132]
plt.plot(data)
plt.title("Streamflow")
plt.show()

**Problems with the vague-prompt output:**
- Uses hardcoded fake data instead of reading from a real source
- No axis labels or units
- No date axis — just integer indices
- Title is generic ("Streamflow") with no site or period information
- No error handling or data validation

### Structured Prompt

> "Using pandas, create a DataFrame with a DatetimeIndex for dates 2023-01-01 through 2023-01-10 and a column 'discharge_cfs' with these daily mean values: [120, 115, 130, 125, 140, 135, 128, 122, 118, 132]. Plot discharge vs. date using matplotlib. Label the x-axis 'Date', the y-axis 'Discharge (cfs)', and title the plot 'USGS 05407000 — Daily Mean Discharge, Jan 2023'. Use a blue line. Add a grid."

#### AI output from the structured prompt:

In [ ]:
# AI-generated code from the STRUCTURED prompt
import pandas as pd
import matplotlib.pyplot as plt

# Build the DataFrame with a proper DatetimeIndex
dates = pd.date_range("2023-01-01", periods=10, freq="D")
discharge_values = [120, 115, 130, 125, 140, 135, 128, 122, 118, 132]
df = pd.DataFrame({"discharge_cfs": discharge_values}, index=dates)

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df["discharge_cfs"], color="blue", linewidth=1.5)
ax.set_xlabel("Date")
ax.set_ylabel("Discharge (cfs)")
ax.set_title("USGS 05407000 \u2014 Daily Mean Discharge, Jan 2023")
ax.grid(True, linestyle="--", alpha=0.7)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Side-by-Side Comparison

| Aspect | Vague Prompt | Structured Prompt |
|--------|-------------|-------------------|
| Data source | Hardcoded list, no context | DataFrame with DatetimeIndex |
| X-axis | Integer indices | Properly formatted dates |
| Y-axis label | None | "Discharge (cfs)" with units |
| Title | Generic "Streamflow" | Site number, parameter, period |
| Grid | No | Yes, with styling |
| Reusability | Low — nothing to adapt | High — clear structure to modify |

**Takeaway:** The structured prompt produced a publication-ready plot on the first try. The vague prompt produced something that would need significant rework. The extra 30 seconds spent writing a detailed prompt saved several minutes of debugging.

---
## 3. Iterative Refinement Demo: Baseflow Separation

Even good prompts sometimes need refinement. This section walks through **three iterations** of refining a prompt for a realistic hydrology task: separating baseflow from total streamflow using a simple digital filter.

The Lyne-Hollick recursive digital filter is:

$$q_f(t) = \alpha \cdot q_f(t-1) + \frac{1 + \alpha}{2} \cdot [Q(t) - Q(t-1)]$$

where $Q(t)$ is total streamflow, $q_f(t)$ is the quickflow component, and baseflow is $q_b(t) = Q(t) - q_f(t)$. The filter parameter $\alpha$ is typically 0.925.

### Iteration 1: First Attempt

**Prompt:**
> "Write a Python function for baseflow separation."

#### AI output:

In [ ]:
# AI output — Iteration 1
def baseflow_separation_v1(streamflow):
    """Separate baseflow from streamflow."""
    baseflow = []
    for i in range(len(streamflow)):
        if i == 0:
            baseflow.append(streamflow[i] * 0.5)
        else:
            bf = 0.925 * baseflow[i-1] + (1 - 0.925) / 2 * (streamflow[i] + streamflow[i-1])
            baseflow.append(bf)
    return baseflow

**What went wrong:**
- The formula is incorrect — it computes baseflow directly instead of computing quickflow first and subtracting
- Uses Python lists instead of NumPy arrays (slow for large datasets)
- No input validation or type hints
- No constraint that baseflow must be non-negative and cannot exceed total streamflow
- The initial condition (50% of first value) is arbitrary

### Iteration 2: Adding the Equation and Data Types

**Refined prompt:**
> "Write a Python function that performs baseflow separation using the Lyne-Hollick recursive digital filter. The quickflow equation is: qf(t) = alpha * qf(t-1) + ((1 + alpha) / 2) * (Q(t) - Q(t-1)), where Q is total streamflow and qf is quickflow. Baseflow is Q(t) - qf(t). Input: a 1D numpy array of daily streamflow in cms. Parameter: alpha (float, default 0.925). Return a 1D numpy array of baseflow values. Use NumPy, not Python lists."

#### AI output:

In [ ]:
# AI output — Iteration 2
import numpy as np

def baseflow_separation_v2(Q: np.ndarray, alpha: float = 0.925) -> np.ndarray:
    """Separate baseflow using the Lyne-Hollick digital filter.

    Parameters
    ----------
    Q : np.ndarray
        1D array of daily streamflow in cms.
    alpha : float
        Filter parameter (default 0.925).

    Returns
    -------
    np.ndarray
        1D array of baseflow values in cms.
    """
    n = len(Q)
    qf = np.zeros(n)  # quickflow
    qf[0] = 0.0  # assume no quickflow at start

    for t in range(1, n):
        qf[t] = alpha * qf[t-1] + ((1 + alpha) / 2) * (Q[t] - Q[t-1])

    baseflow = Q - qf
    return baseflow

In [ ]:
# Test iteration 2
test_Q = np.array([10.0, 12.0, 50.0, 35.0, 20.0, 15.0, 12.0, 11.0, 10.5, 10.0])
bf_v2 = baseflow_separation_v2(test_Q)
print("Total streamflow:", test_Q)
print("Baseflow (v2):   ", np.round(bf_v2, 2))
print("Any negative baseflow?", np.any(bf_v2 < 0))

**What improved:**
- The Lyne-Hollick equation is now correct (quickflow first, then subtract)
- Uses NumPy arrays
- Has type hints and a docstring

**What still needs fixing:**
- Baseflow can go negative (physically impossible — you cannot have negative baseflow)
- Baseflow can exceed total streamflow (also physically impossible)
- No validation that the input array is non-negative

### Iteration 3: Adding Physical Constraints

**Refined prompt:**
> "Update the Lyne-Hollick baseflow separation function with these physical constraints: (1) baseflow must be non-negative — clamp any negative values to zero, (2) baseflow cannot exceed total streamflow at any time step, (3) quickflow must also be non-negative — clamp negative quickflow to zero before computing baseflow. Also validate that the input streamflow array contains no negative values (raise ValueError if it does). Keep the NumPy-style docstring."

#### AI output:

In [ ]:
# AI output — Iteration 3 (final version)
def baseflow_separation_v3(Q: np.ndarray, alpha: float = 0.925) -> np.ndarray:
    """Separate baseflow using the Lyne-Hollick digital filter.

    Applies the recursive digital filter:
        qf(t) = alpha * qf(t-1) + ((1+alpha)/2) * (Q(t) - Q(t-1))
    with physical constraints enforced at each time step.

    Parameters
    ----------
    Q : np.ndarray
        1D array of daily streamflow in cms. Must be non-negative.
    alpha : float
        Filter parameter (default 0.925).

    Returns
    -------
    np.ndarray
        1D array of baseflow values in cms.

    Raises
    ------
    ValueError
        If any streamflow value is negative.
    """
    if np.any(Q < 0):
        raise ValueError("Streamflow values must be non-negative.")

    n = len(Q)
    qf = np.zeros(n)

    for t in range(1, n):
        qf[t] = alpha * qf[t-1] + ((1 + alpha) / 2) * (Q[t] - Q[t-1])
        # Constraint: quickflow cannot be negative
        qf[t] = max(qf[t], 0.0)
        # Constraint: quickflow cannot exceed total streamflow
        qf[t] = min(qf[t], Q[t])

    # Baseflow = total - quickflow (guaranteed non-negative by constraints above)
    baseflow = Q - qf
    return baseflow

In [ ]:
# Test iteration 3
bf_v3 = baseflow_separation_v3(test_Q)
print("Total streamflow:", test_Q)
print("Baseflow (v3):   ", np.round(bf_v3, 2))
print("Any negative baseflow?", np.any(bf_v3 < 0))
print("Baseflow exceeds total?", np.any(bf_v3 > test_Q))

# Verify constraints hold
assert np.all(bf_v3 >= 0), "Baseflow must be non-negative"
assert np.all(bf_v3 <= test_Q), "Baseflow cannot exceed total streamflow"
print("\n\u2705 All physical constraints satisfied.")

# Verify input validation
try:
    baseflow_separation_v3(np.array([10.0, -5.0, 8.0]))
    print("\u274c Should have raised ValueError")
except ValueError as e:
    print(f"\u2705 Correctly rejected negative input: {e}")

### Iteration Summary

| Iteration | What Changed in the Prompt | What Improved in the Output |
|-----------|---------------------------|----------------------------|
| 1 | Vague — "do baseflow separation" | Wrong equation, Python lists, no validation |
| 2 | Added equation, data types, library | Correct equation, NumPy, type hints |
| 3 | Added physical constraints | Non-negative baseflow, input validation |

**Takeaway:** Each iteration added specific, concrete requirements to the prompt. The AI cannot infer physical constraints on its own — you have to state them explicitly. Three iterations took about 2 minutes of prompt writing and produced a robust, physically valid function.

---
## 4. Physical Constraints Exercise

This exercise demonstrates how adding physical constraints to prompts dramatically improves the quality of AI-generated code. We will ask an AI to compute **steady-state 1D groundwater flow** (heads along a confined aquifer) with and without physical constraints.

### Without Physical Constraints

**Prompt:**
> "Write a function to compute hydraulic head along a 1D confined aquifer with constant transmissivity, given head boundary conditions at both ends."

#### AI output:

In [ ]:
# AI output — WITHOUT physical constraints in the prompt
def compute_heads_v1(h_left, h_right, T, length, n_cells):
    """Compute head distribution in a 1D confined aquifer."""
    dx = length / n_cells
    heads = np.linspace(h_left, h_right, n_cells)
    return heads

In [ ]:
# Test without constraints
heads_v1 = compute_heads_v1(h_left=100.0, h_right=90.0, T=-5.0, length=1000.0, n_cells=10)
print("Heads (v1):", np.round(heads_v1, 2))
print(f"Accepted negative T = -5.0 without complaint!")

# Also accepts nonsensical inputs
heads_bad = compute_heads_v1(h_left=100.0, h_right=90.0, T=10.0, length=-500.0, n_cells=10)
print(f"Accepted negative length = -500.0 without complaint!")

**Problems:**
- Accepts negative transmissivity (physically impossible)
- Accepts negative aquifer length (nonsensical)
- No validation of any parameters
- The transmissivity parameter `T` is accepted but never used in the calculation
- No docstring explaining units or constraints

### With Physical Constraints

**Prompt:**
> "Write a Python function that computes steady-state hydraulic head along a 1D confined aquifer using linear interpolation between two fixed-head boundaries (Dirichlet conditions). Parameters: h_left (head at left boundary, meters), h_right (head at right boundary, meters), T (transmissivity in m\u00b2/day, must be positive), length (aquifer length in meters, must be positive), n_cells (number of cells, integer, must be >= 2). Physical constraints: transmissivity must be positive, aquifer length must be positive, and heads must decrease monotonically in the direction of flow (from higher head to lower head). Raise ValueError for invalid inputs. Include a NumPy-style docstring with units."

#### AI output:

In [ ]:
# AI output — WITH physical constraints in the prompt
def compute_heads_v2(
    h_left: float,
    h_right: float,
    T: float,
    length: float,
    n_cells: int,
) -> np.ndarray:
    """Compute steady-state head in a 1D confined aquifer.

    Uses linear interpolation between fixed-head (Dirichlet)
    boundary conditions, which is the analytical solution for
    steady-state flow with constant transmissivity and no recharge.

    Parameters
    ----------
    h_left : float
        Hydraulic head at the left boundary in meters.
    h_right : float
        Hydraulic head at the right boundary in meters.
    T : float
        Transmissivity in m\u00b2/day. Must be positive.
    length : float
        Aquifer length in meters. Must be positive.
    n_cells : int
        Number of cells. Must be >= 2.

    Returns
    -------
    np.ndarray
        1D array of hydraulic head values in meters at cell centers.

    Raises
    ------
    ValueError
        If T or length is not positive, or n_cells < 2.
    """
    if T <= 0:
        raise ValueError(f"Transmissivity must be positive, got {T}")
    if length <= 0:
        raise ValueError(f"Aquifer length must be positive, got {length}")
    if n_cells < 2:
        raise ValueError(f"Need at least 2 cells, got {n_cells}")

    # Cell center positions
    dx = length / n_cells
    x = np.linspace(dx / 2, length - dx / 2, n_cells)

    # Linear interpolation (analytical solution for constant T, no recharge)
    heads = h_left + (h_right - h_left) * (x / length)

    return heads

In [ ]:
# Test with valid inputs
heads_v2 = compute_heads_v2(h_left=100.0, h_right=90.0, T=500.0, length=1000.0, n_cells=10)
print("Heads (v2):", np.round(heads_v2, 2))

# Verify monotonic decrease (flow from left to right)
assert np.all(np.diff(heads_v2) < 0), "Heads should decrease left to right"
print("\u2705 Heads decrease monotonically in flow direction.")

# Verify constraint enforcement
print("\nTesting constraint enforcement:")
for label, kwargs in [
    ("Negative T", dict(h_left=100, h_right=90, T=-5.0, length=1000, n_cells=10)),
    ("Negative length", dict(h_left=100, h_right=90, T=500, length=-100, n_cells=10)),
    ("Too few cells", dict(h_left=100, h_right=90, T=500, length=1000, n_cells=1)),
]:
    try:
        compute_heads_v2(**kwargs)
        print(f"  \u274c {label}: should have raised ValueError")
    except ValueError as e:
        print(f"  \u2705 {label}: {e}")

### Comparison

| Aspect | Without Constraints | With Constraints |
|--------|-------------------|------------------|
| Negative transmissivity | Silently accepted | ValueError raised |
| Negative length | Silently accepted | ValueError raised |
| Parameter validation | None | All parameters checked |
| Docstring with units | No | Yes |
| Type hints | No | Yes |
| T actually used | No (ignored!) | Yes (validated) |

**Takeaway:** Adding physical constraints to the prompt ("transmissivity must be positive", "heads must decrease in the flow direction") caused the AI to generate input validation, proper documentation, and a more robust implementation. The AI does not know physics — you have to encode your domain knowledge into the prompt.

---
## Summary

This notebook covered four key skills for writing effective prompts for scientific programming:

1. **Five Prompt Patterns** — Specifying data types, units/constraints, libraries, example I/O, and documentation requirements
2. **Comparative Exercise** — Seeing how a structured prompt produces dramatically better output than a vague one
3. **Iterative Refinement** — Building up a prompt over multiple iterations, adding specificity each time
4. **Physical Constraints** — Encoding domain knowledge (non-negative K, monotonic heads, mass balance) directly into prompts

**The core principle:** AI assistants are powerful pattern-completion engines, but they have zero understanding of hydrology. Every physical constraint, unit specification, and data format requirement that you leave out of your prompt is something the AI will guess — and it will often guess wrong. Invest the extra minute in writing a detailed prompt, and you will save many minutes of debugging.